# Öğrenci Ayrımı ve Eğitim Verisinden Özellikler

Bu defter veri hazırlama aşamasını yeniden çalıştırır. Öğrenciler ayrılır; kimlik sözlükleri, soru zorluğu ve sabit referans soruları yalnızca eğitim bölümünden belirlenir. Model eğitimi bu defterin kapsamı değildir.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kt_project").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "kt_project").exists()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

## Veri hazırlamayı yeniden çalıştırma

Çok becerili bir gerçek cevap tek etkileşim olarak tutulur. Öğrenciler yüzde 70 eğitim, yüzde 15 doğrulama ve kalan yüzde 15 test olacak şekilde ayrılır. Ayrım başlangıcı 2026 olarak sabittir.

In [2]:
from kt_project.prepare_data import prepare_dataset

metadata = prepare_dataset(
    ROOT / "data" / "raw" / "skill_builder_data_corrected_collapsed.csv",
    ROOT / "data" / "processed", split_seed=2026)
table = pd.DataFrame(metadata["splits"]).T
table.index.name = "Grup"
display(table.rename(columns={
    "students": "Öğrenci", "interactions": "Etkileşim",
    "unseen_question_interactions": "Eğitimde görülmeyen soru içeren etkileşim",
    "interactions_with_unseen_skill": "Eğitimde görülmeyen beceri içeren etkileşim"}))

,Öğrenci,Etkileşim,Eğitimde görülmeyen soru içeren etkileşim,Eğitimde görülmeyen beceri içeren etkileşim
Grup,,,,
train,2550,197213,0,0
validation,546,39338,548,0
test,548,45339,308,0


## Öğrenci kümelerinin ayrılığını kontrol etme

Aynı öğrenciye ait kayıtlar farklı gruplara dağılmaz. Test öğrencilerinin erken cevapları, o öğrencinin sonraki tahminleri için gözlenen geçmiş olabilir; model ağırlıklarını eğitmek için kullanılamaz.

In [3]:
prepared = ROOT / "data" / "processed"
splits = json.loads((prepared / "student_splits.json").read_text(encoding="utf-8"))
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not set(splits[left]).intersection(splits[right])
    print(left, "ve", right, "öğrenci kesişimi: 0")

train ve validation öğrenci kesişimi: 0
train ve test öğrenci kesişimi: 0
validation ve test öğrenci kesişimi: 0


## Zorluğun yalnızca eğitimden hesaplandığını kontrol etme

Bilinen sorunun zorluğu eğitimdeki hata oranıdır. Eğitimde bulunmayan soru için genel eğitim hata oranı kullanılır. Öğrenci bazında ayırma nedeniyle ortaya çıkan bu eksik kimlik durumu, yeni soru bankası geliştirmekten farklıdır ve raporda ayrıca belirtilir.

In [4]:
with np.load(prepared / "interactions.npz", allow_pickle=False) as packed:
    arrays = {name: packed[name] for name in packed.files}
training = arrays["split"] == 0
training_error = float(1-arrays["correct"][training].mean())
assert np.isclose(training_error, metadata["global_training_difficulty"])
assert not (arrays["question"][training] == 1).any()
unknown = arrays["question"] == 1
assert np.allclose(arrays["difficulty"][unknown], training_error)
display(pd.DataFrame([{
    "Eğitimden hesaplanan genel hata oranı": training_error,
    "Tüm etkileşimler": len(arrays["correct"]),
    "Bir olayda en fazla beceri": metadata["max_skills_per_event"],
}]))

,Eğitimden hesaplanan genel hata oranı,Tüm etkileşimler,Bir olayda en fazla beceri
0,0.339389,281890,4


## Sabit referans sorularının kapsamı

Her beceri için eğitim sorularından, farklı zorlukları kapsayan en fazla 32 referans soru seçilir. Göstergenin değişimi öğrenci geçmişinden kaynaklanır; referans kümesi oturumda değiştirilmez.

In [5]:
references = json.loads((prepared / "references.json").read_text(encoding="utf-8"))
counts = pd.Series({skill: len(items) for skill, items in references.items()}, dtype="int64")
display(counts.describe().to_frame("Beceri başına referans soru sayısı"))
train_questions = set(arrays["question"][training].tolist())
assert all(item["question"] in train_questions
           for items in references.values() for item in items)
print("Referans beceri sayısı:", len(references))
print("Bütün referans sorular eğitim sorularından seçilmiş.")

,Beceri başına referans soru sayısı
count,123.000000
mean,25.756098
std,10.829030
min,1.000000
25%,26.000000
50%,32.000000
75%,32.000000
max,32.000000


Referans beceri sayısı: 123
Bütün referans sorular eğitim sorularından seçilmiş.


## Eğitimde görülmeyen sorular

Bu etkileşimler testten sessizce çıkarılmaz. Soru için UNK kimliği, olayda verilen bilinen beceriler ve eğitimden hesaplanan genel zorluk kullanılır. Sonuçlarda tüm örnekler ve bilinen soru alt grubu ayrı gösterilecektir.

In [6]:
coverage = pd.DataFrame(metadata["splits"]).T
coverage["Eğitimde görülmeyen soru oranı yüzde"] = (
    100 * coverage["unseen_question_interactions"] / coverage["interactions"])
display(coverage[["Eğitimde görülmeyen soru oranı yüzde"]].round(3))
print("Hazırlanan dosyalar: interactions.npz, metadata.json, references.json, vocabulary.json, student_splits.json")

,Eğitimde görülmeyen soru oranı yüzde
train,0.000
validation,1.393
test,0.679


Hazırlanan dosyalar: interactions.npz, metadata.json, references.json, vocabulary.json, student_splits.json


## Sonraki aşama

Hazırlanan dizilerle küçük model denemesi yapılacak; hedef cevabın gizlenmesi, geleceğe erişimin engellenmesi ve prompt gradyanları kontrol edilecek. Ardından LKM ön eğitimi ve üç yöntem için bağımsız başlangıçlarla ana deneyler yürütülecek.